# Task 2: Qwen3-ASR-1.7B (CPU version)
Run the 3 cells in order. Runs on CPU, no GPU needed. It is slow, so wait for each cell to finish.

In [1]:
!pip install -q -U qwen-asr jiwer datasets soundfile psutil

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import torch, time, psutil, os
from qwen_asr import Qwen3ASRModel
t = time.time()
model = Qwen3ASRModel.from_pretrained("Qwen/Qwen3-ASR-1.7B", dtype=torch.float32, device_map="cpu")
print("load time:", round(time.time()-t,1), "s")
print("CPU cores:", os.cpu_count(), "| RAM used GB:", round(psutil.Process().memory_info().rss/1e9,2))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/478M [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.22G [00:00<?, ?B/s]

... (5 outputs left out)

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

load time: 104.2 s
CPU cores: 2 | RAM used GB: 9.29


In [3]:
import re, jiwer, soundfile as sf
from datasets import load_dataset
fix = lambda t: re.sub(r"[^\w\s]", "", t.lower())
for kind in ["clean", "other"]:
    for ex in load_dataset("openslr/librispeech_asr", kind, split="test", streaming=True).take(2):
        sf.write("a.wav", ex["audio"]["array"], 16000)
        sec = len(ex["audio"]["array"]) / 16000
        start = time.time()
        hyp = model.transcribe(audio="a.wav")[0].text
        took = time.time() - start
        print(kind, "| audio", round(sec,1), "s | time", round(took,2), "s | WER", round(jiwer.wer(fix(ex["text"]), fix(hyp)),3), "| CER", round(jiwer.cer(fix(ex["text"]), fix(hyp)),3))
        print("  REF:", ex["text"].lower()); print("  HYP:", hyp)

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


clean | audio 3.5 s | time 43.23 s | WER 0.0 | CER 0.0
  REF: concord returned to its place amidst the tents
  HYP: Concord returned to its place amidst the tents.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


clean | audio 14.2 s | time 52.71 s | WER 0.0 | CER 0.0
  REF: the english forwarded to the french baskets of flowers of which they had made a plentiful provision to greet the arrival of the young princess the french in return invited the english to a supper which was to be given the next day
  HYP: The English forwarded to the French baskets of flowers, of which they had made a plentiful provision to greet the arrival of the young princess. The French, in return, invited the English to a supper, which was to be given the next day.


Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/64 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


other | audio 2.1 s | time 14.66 s | WER 0.222 | CER 0.05
  REF: i am from the cutter lying off the coast
  HYP: I'm from the cutter lying off the coast.
other | audio 2.5 s | time 15.77 s | WER 0.0 | CER 0.0
  REF: don't cry he said i was obliged to come
  HYP: Don't cry," he said. "I was obliged to come."
